In [38]:
import pandas as pd
import numpy as np
file_path = 'C:/Users/md.shamim/source/IITM/StudyMaterial/Python_Assignment/Day4/patient_clinical_data_raw.csv'
df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("-" * 50)

print(df.head(10))

print("\nLast 10 Records:")
print(df.tail(10))

print("-" * 50)

rows, columns = df.shape

print(f"Number of rows    : {rows}")
print(f"Number of columns : {columns}")

print("Column Names and Data Types:")
print(df.dtypes)

print("-" * 50)

# Alternatively, display them in a cleaner table
column_info = pd.DataFrame({
    "Column Name": df.columns,
    "Data Type": df.dtypes.astype(str)
})

print(column_info)

print("-" * 50)

print("Statistical Summary of Numerical Columns:")
print(df.describe())

print("-" * 50)

print("Number of Unique Values in Each Column:")
print(df.nunique())

print("-" * 50)

print("Columns containing missing values:")
missing_columns = df.columns[df.isnull().any()]

print(missing_columns.tolist())

# Also display the number of missing values
print("\nNumber of missing values in each column:")
print(df.isnull().sum())

missing_percentage = (df.isnull().sum() / len(df)) * 100

print("\nPercentage of missing values in each column:")
print(missing_percentage)

print("\nColumns with missing values and their percentages:")
print(
    missing_percentage[missing_percentage > 0]
    .sort_values(ascending=False)
)

# Identify numerical columns
numerical_columns = df.select_dtypes(
    include=["int64", "float64"]
).columns


print("\nNumerical columns:")
print(numerical_columns.tolist())

# Fill missing numerical values with the median
# Median is less affected by extreme/outlier values.
for column in numerical_columns:
    if df[column].isnull().any():
        df[column] = df[column].fillna(df[column].median())

print("\nMissing values after handling numerical columns:")
print(df[numerical_columns].isnull().sum())


print("\nNumerical columns:")
print(numerical_columns.tolist())

# Fill missing numerical values with the median
# Median is less affected by extreme/outlier values.
for column in numerical_columns:
    if df[column].isnull().any():
        df[column] = df[column].fillna(df[column].median())

print("\nMissing values after handling numerical columns:")
print(df[numerical_columns].isnull().sum())

duplicate_count = df.duplicated().sum()

print(f"\nNumber of duplicate records: {duplicate_count}")

# Remove duplicate records
df = df.drop_duplicates()

print(f"Number of records after removing duplicates: {len(df)}")

print("\nAge column statistics:")
print(df["Age"].describe())
# Display ages outside a reasonable range
invalid_age = df[(df["Age"] < 0) | (df["Age"] > 120)]

print("\nPatients with invalid ages:")
print(invalid_age[["Age"]])
# Replace invalid ages with NaN
df.loc[(df["Age"] < 0) | (df["Age"] > 120), "Age"] = np.nan

# Fill invalid/missing ages with the median age
df["Age"] = df["Age"].fillna(df["Age"].median())

print("\nAge column after handling invalid values:")
print(df["Age"].describe())

# Check unique values in Gender
print("\nUnique values in Gender:")
print(df["Gender"].unique())

# Check unique values in Department
print("\nUnique values in Department:")
print(df["Department"].unique())
df["Gender"] = df["Gender"].astype(str).str.strip()
df["Department"] = df["Department"].astype(str).str.strip()

df["Gender"] = df["Gender"].str.lower()

df["Gender"] = df["Gender"].replace({
    "m": "Male",
    "male": "Male",
    "man": "Male",
    "f": "Female",
    "female": "Female",
    "woman": "Female"
})

# Standardize Department values
df["Department"] = df["Department"].str.lower()

df["Department"] = df["Department"].replace({
    "cardiology": "Cardiology",
    "cardio": "Cardiology",
    "neurology": "Neurology",
    "neuro": "Neurology",
    "orthopedics": "Orthopedics",
    "orthopaedics": "Orthopedics",
    "orthopedic": "Orthopedics",
    "pediatrics": "Pediatrics",
    "paediatrics": "Pediatrics",
    "general medicine": "General Medicine",
    "general_medicine": "General Medicine",
    "generalmedicine": "General Medicine"
})

print("\nStandardized Gender values:")
print(df["Gender"].unique())

print("\nStandardized Department values:")
print(df["Department"].unique())


# --------------------------------------------------
# Final check after cleaning
# --------------------------------------------------

print("\nFinal dataset shape:")
print(df.shape)

print("\nRemaining missing values:")
print(df.isnull().sum())

print("\nRemaining duplicate records:")
print(df.duplicated().sum())

df["Age_Group"] = pd.cut(
    df["Age"],
    bins=[-np.inf, 17, 40, 60, np.inf],
    labels=[
        "Child",
        "Young Adult",
        "Middle Age",
        "Senior"
    ]
)

print("\nAge Group:")
print(df["Age_Group"].value_counts().sort_index())

df[["Systolic", "Diastolic"]] = (
    df["Blood_Pressure"]
    .str.split("/", expand=True)
    .astype(int)
)

print("\nBlood Pressure:")
print(df[["Blood_Pressure", "Systolic", "Diastolic"]].head())


df["High_Risk"] = np.where(
    (
        (df["Systolic"] >= 140) |
        (df["Diastolic"] >= 90) |
        (df["Glucose"] >= 126) |
        (df["Cholesterol"] >= 240) |
        (df["Heart_Rate"] > 100) |
        (df["Temperature"] >= 100.4) |
        (df["Diagnosis"].isin([
            "Diabetes",
            "Hypertension",
            "Heart Disease"
        ]))
    ),
    "Yes",
    "No"
)
print("\nHigh Risk:")
print(df["High_Risk"].value_counts())

df["Cost_Per_Day"] = (
    df["Treatment_Cost"] / df["Length_of_Stay"]
)

print("\nCost Per Day:")
print(df[
    ["Patient_ID", "Treatment_Cost",
     "Length_of_Stay", "Cost_Per_Day"]
].head())


average_age = df["Age"].mean()

print("\n19. Average Age:")
print(round(average_age, 2))

department_count = df["Department"].value_counts()

print("\n20. Patients per Department:")
print(department_count)


department_cost = (
    df.groupby("Department")["Treatment_Cost"]
      .mean()
      .sort_values(ascending=False)
)

print("\n21. Average Treatment Cost by Department:")
print(department_cost.round(2))

diagnosis_los = (
    df.groupby("Diagnosis")["Length_of_Stay"]
      .mean()
      .sort_values(ascending=False)
)

print("\n22. Average Length of Stay by Diagnosis:")
print(diagnosis_los.round(2))


top_10 = (
    df.nlargest(10, "Treatment_Cost")
    [["Patient_ID",
      "Department",
      "Diagnosis",
      "Treatment_Cost"]]
)

print("\n23. Top 10 Patients by Treatment Cost:")
print(top_10)


highest_cost_department = (
    df.groupby("Department")["Treatment_Cost"]
      .mean()
      .idxmax()
)

highest_cost = (
    df.groupby("Department")["Treatment_Cost"]
      .mean()
      .max()
)

print("\n24. Department with Highest Average Treatment Cost:")
print(highest_cost_department)
print("Average Cost:", round(highest_cost, 2))


readmission_percentage = (
    df["Readmission"].eq("Yes").mean() * 100
)

print("\n25. Readmission Percentage:")
print(round(readmission_percentage, 2), "%")

readmission_by_department = (
    pd.crosstab(
        df["Department"],
        df["Readmission"],
        normalize="index"
    )["Yes"] * 100
)

print("\n26. Readmission Rate by Department:")
print(readmission_by_department.round(2))

clinical_by_diagnosis = (
    df.groupby("Diagnosis")
      [["Cholesterol", "Glucose"]]
      .mean()
)

print("\n27. Average Cholesterol and Glucose by Diagnosis:")
print(clinical_by_diagnosis.round(2))


high_glucose = df[df["Glucose"] > 126]

high_cholesterol = df[df["Cholesterol"] > 240]

high_heart_rate = df[df["Heart_Rate"] > 100]

print("\n28. High Glucose Patients:")
print(high_glucose[
    ["Patient_ID", "Glucose"]
])

print("\nHigh Cholesterol Patients:")
print(high_cholesterol[
    ["Patient_ID", "Cholesterol"]
])

print("\nHigh Heart Rate Patients:")
print(high_heart_rate[
    ["Patient_ID", "Heart_Rate"]
])


Dataset loaded successfully!
--------------------------------------------------
  Patient_ID   Age  Gender        Department Blood_Pressure  Heart_Rate  \
0     P07402  58.0  Female       Orthopedics         119/73        64.0   
1     P05835   NaN  Female  General Medicine         107/91        62.0   
2     P02123  47.0    Male       Orthopedics         148/98        83.0   
3     P08789   NaN    Male          Oncology         115/84        89.0   
4     P00305  47.0  Female       Orthopedics         128/62        77.0   
5     P02532  31.0  Female          Oncology         104/85        99.0   
6     P02996  49.0    MALE         Neurology         118/83       108.0   
7     P07660  32.0  Female        Pediatrics         122/89        64.0   
8     P08225  64.0    Male         Neurology        161/100        61.0   
9     P04449  56.0    Male  General Medicine         102/76        83.0   

   Temperature  Cholesterol  Glucose      Diagnosis Admission_Type  \
0        100.0        19